# **Тема:** RAG (Retrieval-Augmented Generation) с фреймворком LangChain


Разработка RAG-пайплайна


**Задачи:**
* Загрузить набор текстовых документов (например, статей из датасета arXiv Dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv)
* Разбить текст на чанки с помощью Langchain text splitter
* Создать векторный индекс с помощью FAISS и sentence-transformers
* Реализовать langchain-цепочку, которая производим семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)
* Протестировать систему на нескольких вопросах, оценить качество ответов


**Библиотеки:** langchain, huggingface, faiss-cpu, sentence-transformers

**Ожидаемый результат:** Colab-ноутбук с рабочим прототипом наукоёмкой (например, разработанной на основе текстов ArXiv) RAG-системы, примерами её ответов и качественным анализом, представленным в текстовых блоках


Группа: Иванюк М., Липатова Д., Туманова А.

## Загрузить набор текстовых документов

### Датасет с метаданными к статьям

In [1]:
!pip install --upgrade huggingface_hub langchain-huggingface llama_index -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [3]:
# путь к файлу датасета
file_path = 'arxiv-metadata-oai-snapshot.json'

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "Cornell-University/arxiv",
  file_path,
  pandas_kwargs={"lines": True, "nrows": 10000}, # указываем число строк
)
df["id"] = df["id"].apply(lambda x: str(x).zfill(9))

/tmp/ipykernel_1076/2813351605.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'arxiv' dataset.


In [4]:
# заглянем
df.head(2)

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"


### Подбор статей

внимание на столбец id: по нему можно добраться до самой статьи

 https://arxiv.org/abs/{id}: посмотреть страницу статьи с абстрактом

 https://arxiv.org/pdf/{id}: скачать

Так как в arXiv статьи на разные темы, нужно как-то тематически ограничить те статьи, которые будут использоваться в RAG системе. Мы решили выбрать статьи, в аннотации которых упоминаются графы (graph/graphs). Так, задавая вопросы про графы, можно с большей уверенностью ожидать, что у системы будет ответ на этот вопрос. В итоге из 10000 статей были отобраны 210. Из-за ошибок загрузки части статей, в итоговую версию попали 187.

In [5]:
graphs = df[df['abstract'].str.contains(r'\bgraphs?\b', case=False)]
len(graphs)

210

In [6]:
# отфильтруем df по заданному признаку и получим список id для дальнейшего сохранения

ids = list(graphs['id'] )
ids[:2]

['0704.0002', '00704.001']

### Загрузка

In [7]:
!pip install langchain_community langchain_text_splitters pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import re

Чистка неидеальная: возможно из-за исправления слов, перенесенных на другую строку (как в situa-tion), слова, в которых должен быть дефис но они попали на перенос строки, будут лишены дефиса

Все переносы строк были заменены на пробелы, так как лишних переносов гораздо больше, чем нужных, что может негативно сказаться при разбиении на чанки

Некоторые специальные символы обозначали переменные, поэтому в некоторых местах теряется важная для содержания информация

Основной целью было сохранить целостность слов и предложений, и в целом кажется, что это удалось

In [9]:
objects = []

for id in ids:
    try:
        loaded = PyPDFLoader(f'https://arxiv.org/pdf/{id}', mode="single").load()[0] # Загружаем статьи одной страницей, чтобы было удобнее делить на чанки
        raw_text = loaded.page_content
        text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', raw_text)                           # Убираем разрывы внутри слов
        text = re.sub(r'\n', ' ', text)                                              # Заменяем переносы строк на пробелы
        text = re.sub(r'^.*?\bAbstract\b|\bReferences\b.*$', '', text)               # Удаляем все до аннотации и список литературы
        text = re.sub(r'\[\d+\]', '', text)                                          # Удаляем сноски
        text = re.sub(r'https?://\S+', '', text)                                     # Удаляем ссылки
        text = re.sub(r'\S+@\S+\.\S+', '', text)                                     # Удаляем email-адреса
        text = re.sub(r'[^a-zA-Z0-9\s.,!?;:()-]', '', text)                          # Убираем спецсимволы
        text = re.sub(r'\s+', ' ', text)                                             # Исправляем пробелы
        objects.append(Document(page_content=text, metadata=loaded.metadata))
    except:
        print(f'Can\'t load file: {id}')

Can't load file: 00704.001
Can't load file: 0704.0644
Can't load file: 00704.102
Can't load file: 00704.131
Can't load file: 00704.179
Can't load file: 000704.26
Can't load file: 00704.265
Can't load file: 0704.3696
Can't load file: 000705.01
Can't load file: 00705.014
Can't load file: 00705.076
Can't load file: 000705.11
Can't load file: 00705.262
Can't load file: 00705.369
Can't load file: 00705.382
Can't load file: 00706.012
Can't load file: 00706.043
Can't load file: 00706.044
Can't load file: 0706.0548
Can't load file: 00706.063
Can't load file: 00706.068
Can't load file: 00706.076
Can't load file: 0706.1002


## Разбить на чанки

Выбран RecursiveCharacterTextSplitter, так как при нем слова не должны дробиться, стремится сохранить предложения и абзацы

Размер одного чанка 500 - должно влезать по несколько предложений, overlap 20% для сохранения контекста

Изменены разделители, чтобы делилось по предложениям (по точкам с пробелами), чтобы не было возможности отрезать куски предложений

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". "],
    keep_separator=False,
    chunk_size=500,
    chunk_overlap=100
)

# Разбиваем текст
texts = [obj.page_content for obj in objects]
metadatas = [obj.metadata for obj in objects]

chunks = text_splitter.create_documents(
    texts=texts,
    metadatas=metadatas
)

In [11]:
len(chunks)

18178

## Создать векторный индекс с помощью faiss-cpu и sentence-transformers

In [12]:
!pip install faiss-cpu sentence-transformers langchain-huggingface -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.2 MB/s eta 0:00:00


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

Выбрана модель all-MiniLM-L6-v2, подходящая для английского языка, с размером вектора 384, что нам кажется достаточным для векторизации небольших абзацев, которые представляют из себя получившиеся у нас чанки

In [14]:
# выбрать модель
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Обработано чанков: {vectorstore.index.ntotal}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Обработано чанков: 18178


## Реализовать цепочку

In [15]:
!pip install langchain_core langchain_classic langchain_openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.8/505.8 kB 24.2 MB/s eta 0:00:00


In [16]:
import json
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from google.colab import userdata
from langchain_openai import ChatOpenAI

В качестве модели была взята бесплатная модель arcee-ai/trinity-large-preview:free, использовавшаяся в рамках семинара. Она подходит для английского языка и выполняемого задания.

In [17]:
# задаем параметры
GEN_MODEL_ID = "arcee-ai/trinity-large-preview:free"
OPEN_ROUTER_API_KEY = userdata.get('OPEN')
TOP_K = 3
QUESTION = "What is an oriented graph?"
PROMPT = PromptTemplate(
    input_variables=["context", "input"],
    template="""Answer the following question based on the context:
    Question: {input}
    Context: {context}
    Answer:
    If there is not enough information in the context, don't answer by yourself, just say that it's not presented, but what there is similar to the question"""
)
TASK = "text-generation"

LLMки отчаянно отказывались подгружаться через hugging face, поэтому после долгих попыток было принято решение подгрузить через OpenRouter

Выбран k, равный 3, чтобы у модели была возможность посмотреть на несколько контекстов перед выдачей ответа, и вместе с тем не получать в контексте менее релевантные тексты.

In [18]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

llm = ChatOpenAI(
    api_key=OPEN_ROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    model=GEN_MODEL_ID, # бесплатная модель
)

In [19]:
def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [20]:
question_answer_chain = create_stuff_documents_chain(llm, PROMPT)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

def get_answer(question=QUESTION):
    resp_dict = rag_chain.invoke({"input": question})

    clipped_answer = clip_text(resp_dict["answer"], threshold=350)
    print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")
    for i, doc in enumerate(resp_dict["context"]):
        print()
        print(f"Source {i + 1}:")
        print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
        for key in doc.metadata:
            if key != "pk":
                val = doc.metadata.get(key)
                clipped_val = clip_text(val) if isinstance(val, str) else val
                print(f"  {key}: {clipped_val}")

get_answer()

Question:
What is an oriented graph?

Answer:
An oriented graph, as described in the context, is a multi-graph (which allows loops and multiple edges) embedded in an oriented surface. The embedding ensures that the complement of the graph in the surface is a union of 2-cells. This embedding determines a cyclic order on the edges at every vertex, which is referred to as the orientation of the r...

Source 1:
  text: "In the case when the surface is oriented, the embeddin g determines a cyclic order on the edges at every vertex, which is called an orientation fo r the ribbon graph. Other terms for oriented ribbon graphs include: combinatorial map s, fat graphs, cyclic graphs, graphs with rotation systems, and dessins denfant (see ). In this paper, all ribbon gra..."
  producer: dvips + GPL Ghostscript GIT PRERELEASE 9.22
  creator: LaTeX with hyperref package
  creationdate: 2018-10-30T10:31:13-04:00
  moddate: 2018-10-30T10:31:13-04:00
  title: 
  subject: 
  author: 
  keywords: 
  sou

## Протестировать на нескольких примерах, оченить качество



---



In [21]:
for i in ['What is a graph?', 'What kinds of graphs exist?', 'What can graphs be used for?']:
    get_answer(i)
    print('\n\n')

Question:
What is a graph?

Answer:
The context does not provide a direct definition of what a graph is. However, it does describe graphs as a formal structure used to represent networks, where vertices (or nodes) stand for units like genes, proteins, cells, or neurons, and edges represent correlations or interactions between these units. The edges can be directed or weighted to enco...

Source 1:
  text: "The vertices of a graph stand for the units in question, like genes, proteins, cells, neurons, and an edge between vertices expresses some correlation or interaction between the corresponding units. These edges can be directed, to encode the direction of interaction, for example via a synaptic connection between neurons, and weighted, to express th..."
  producer: pdfTeX-1.40.4
  creator: LaTeX with hyperref package
  creationdate: 2018-10-23T14:17:46+00:00
  author: 
  keywords: 
  moddate: 2018-10-23T14:17:46+00:00
  ptex.fullbanner: This is pdfTeX, Version 3.141592-1.40.5-2.2 (Web

Ответы на вопросы:
* What is a graph? - Модель отвечает, что точно подходящей информации в контексте нет, но есть что-то похожее и предлагает варианты. Дальнейший ответ получается правдивым только для конкретной области. В контекст попадают чанки, в которых есть что-то похожее на определение графа, его части или подвида (The vertices of a graph stand for...; ...represented as networks, that is, in terms of the formal structure of a graph; Graph G is called a cycle if...).
* What kinds of graphs exist? - Модель также говорит, что прямого ответа на вопрос в контексте нет, далее перечисляет подходящие варианты из контекста. В контекст, вероятно, попадают чанки, в которых что-то говорится про подвиды графов (...common to large classes of graphs...; ...the species of graphs...; These graphs...)
* What can graphs be used for? - Здесь модель отвечает на вопрос сразу, но в контекст попадают только части одной статьи. В чанках содержится информация о способах использования графов или их частей (analysis of the represented biological structures; to encode the direction of interaction...; graph almost fully represents...)

В целом, система работает, отвечает на вопросы по имеющейся информации. Но кажется, что для улучшения результатов нужно дополнить датасет: помимо статей, количество которых хотелось бы увеличить, в него необходимо добавить учебные материалы, чтобы ответы на самые базовые вопросы точно были. Также было бы полезно более аккуратно чистить тексты, сохраняя абзацы, некоторые символы, отвечающие за переменные, и др.

# Критерии оценки

Работа проверяется по следующим критериям (максимум 10 баллов):

### Загрузка и подготовка данных (2 балла)
- [ ] 0.5 балла: выбран критерий подбора материалов
- [ ] 0.5 балла: загружено не менее 100 записей/статей
- [ ] 0.5 балла: тексты успешно извлечены из источника
- [ ] 0.5 балла: данные приведены к формату, пригодному для чанкинга (очистка, объединение полей)

### Чанкинг (2 балла)
- [ ] 0.5 балла: выбран подходящий тип сплиттера (RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter и т.д.)
- [ ] 0.5 балла: обоснован выбор размера чанка и перекрытия (например, "512 токенов, overlap 20% для сохранения контекста")
- [ ] 0.5 балла: чанки созданы и не содержат явных артефактов (оборванных слов)
- [ ] 0.5 балла: количество чанков соответствует ожидаемому (не 1 и не 100500 на документ)

### Векторное хранилище (1 балл)
- [ ] 0.5 балла: выбрана адекватная эмбеддинг-модель (например, all-MiniLM-L6-v2 для русского/английского)
- [ ] 0.5 балла: индекс создан


### Реализация цепочки (3 балла)
- [ ] 0.5 балла: выбрана LLM
- [ ] 1 балл: промпт, QUESTION, TASK составлены корректно
- [ ] 0.5 балла: обоснован заданный TOP_K
- [ ] 0.5 балла: ответ генерируется на основе найденных чанков (видно по содержанию)
- [ ] 0.5 балла: обработан случай отсутствия информации в контексте

### Тестирование и анализ (2 балла)
- [ ] 0.5 балла: задано минимум 3 разнотипных вопроса (фактический, обобщающий, уточняющий)
- [ ] 0.5 балла: для каждого вопроса показан и проанализирован ответ
- [ ] 0.5 балла: в анализе указано, какие чанки использовались и почему
- [ ] 0.5 балла: сделан вывод о качестве работы системы (что получилось, что нет, гипотезы почему)
